# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — FAIR² Dataset Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process a Croissant-structured dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is defined by a Croissant schema JSON-LD and accessible online, supporting reproducible, programmatic exploration and processing.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access and display dataset metadata
metadata = dataset.metadata.to_json()
print(f"{metadata.get('name', 'Unnamed Dataset')}: {metadata.get('description', 'No description.')}")

### Dataset high-level info
- **ID:** https://api.app.sen.science/frontiers/7853015/6fa2b1a4-3434-4385-9e28-d2d1a49919f2
- **Identifier:** 10.71728/senscience.y7m0-f273
- **Date Published:** 2026-07-30
- **Spatial Coverage:** Samburu, Isiolo, Marsabit counties, Northern Kenya

## 2. Data Overview
Review available record sets, their `@id` values, fields, and discover structure for later programmatic referencing.

In [ ]:
# List each record set, its `@id` and fields' `@id`.
print('Record Sets and their Fields:')
record_sets = dataset.record_sets
record_set_ids = []
for rs in record_sets:
    rs_id = rs['@id']
    record_set_ids.append(rs_id)
    print(f"\nRecord Set @id: {rs_id}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        # Fields are referenced as @id objects (or strings)
        field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
        print(f"  Field @id: {field_id}")

> **Note:** All further data access will reference record sets and fields by their `@id` for consistency and transparency.

## 3. Data Extraction
Load records for each record set into DataFrames, referencing all record sets and fields by their `@id`.

In [ ]:
# Extract data for each record set into a DataFrame (by @id)
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records for Record Set @id: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for {record_set_id}: {df.shape[0]} rows, {df.shape[1]} columns.")
            print("Columns (@id):", df.columns.tolist())
            display(df.head())
        else:
            print("No records found.")
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {repr(e)}")

#### Choose one record set for EDA below.

- Use a record set `@id` printed above. For demonstration, we pick the first non-empty one.

In [ ]:
# Pick the first non-empty record set for EDA
for rs_id, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rs_id
        break
else:
    raise ValueError("No non-empty record sets available.")
print(f"Selected main record set for EDA: {main_record_set_id}")
print(f"Columns (@id): {dataframes[main_record_set_id].columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)
Apply common processing: filter numeric fields, normalize, and group. All columns referenced by their `@id`.

In [ ]:
# List numeric fields by inspecting dtypes (likely: log_likelihood, coefficient, standard_error, p_value, etc)
df = dataframes[main_record_set_id]
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields: {numeric_fields}")

# Sanity check: Pick a numeric field and a group field by @id
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    raise ValueError("No numeric fields detected.")

# Try to pick a grouping field (often categorical, string type)
category_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) and col != numeric_field_id]
group_field_id = category_fields[0] if category_fields else None

print(f"Using numeric field: {numeric_field_id}")
if group_field_id:
    print(f"Grouping by field: {group_field_id}")
else:
    print("No grouping field detected.")

# Example filtering (e.g., for log_likelihood or coefficient > threshold)
threshold = df[numeric_field_id].quantile(0.75)  # Use 75th percentile as threshold
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"\nFiltered records with {numeric_field_id} > {threshold:.3f}:")
display(filtered_df.head())

# Normalize numeric field in filtered records
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Group and aggregate
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
    print(f"\nGrouped data by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Visualize numeric field distribution and relationships by group using `matplotlib`/`seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the chosen numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# If grouping, show boxplot
if group_field_id:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
In this notebook, you've:
- Loaded a FAIR² dataset described by a Croissant schema using `mlcroissant`.
- Explored the dataset's structure via record sets and fields, referencing all elements by their `@id`.
- Loaded and visualized records in DataFrames, filtered and normalized numeric fields, and examined group-wise summaries.

This workflow demonstrates reproducible and programmatic data exploration for structured FAIR datasets with strong schema referencing. For further studies, you can: merge across record sets (using matching `@id` fields), perform machine learning analysis, or extend visualizations according to research questions.
